# Okapi BM25 Ranking Model

---

## BM25 Formula

For a query $Q$ with terms $q_1, q_2, ..., q_n$ and a document $D$:

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Where:
- $f(q_i, D)$ = term frequency of $q_i$ in document $D$
- $|D|$ = length of document $D$ (total number of terms)
- $\text{avgdl}$ = average document length across the corpus
- $k_1 = 1.6$ (term frequency saturation parameter)
- $b = 0.75$ (length normalization parameter)

### IDF Formula (Robertson-Sparck Jones)

$$\text{IDF}(q_i) = \ln\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)$$

Where:
- $N$ = total number of documents
- $n(q_i)$ = number of documents containing term $q_i$

---
## Step 1: Imports

In [1]:
import numpy as np
import pandas as pd
import math

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries loaded successfully!")

Libraries loaded successfully!


---
## Step 2: Define the Corpus and Query

We define a corpus of **20 documents** over a **vocabulary of 10 terms**.

- Terms like `"data"`, `"model"`, and `"search"` are **common** (appear in many documents)
- Terms like `"neural"`, `"quantum"`, `"sparse"` are **rare** (appear in only a few documents)
- Document lengths vary significantly to showcase BM25's length normalization

**Query:** `"data neural search model"` (4 terms)

In [2]:
# ── Vocabulary (10 terms) ──────────────────────────────────────────────────
terms = ["data", "model", "search", "rank", "neural", "query", "index", "sparse", "bert", "quantum"]

# ── Term-frequency matrix: 20 documents × 10 terms ────────────────────────
# Rows = documents, Columns = terms
# Some terms appear in all/most docs (data, model, search); others are rare (neural, quantum, bert)
tf_matrix = [
#  data  model search rank neural query index sparse bert quantum
  [5,    3,    2,    1,   0,     2,    0,    0,     0,   0   ],  # D1  — short doc
  [3,    2,    4,    0,   1,     1,    2,    0,     0,   0   ],  # D2
  [1,    4,    1,    2,   0,     3,    1,    1,     0,   0   ],  # D3
  [7,    1,    3,    0,   2,     0,    0,    0,     1,   0   ],  # D4
  [2,    5,    0,    3,   0,     4,    2,    0,     0,   0   ],  # D5
  [4,    3,    5,    1,   3,     2,    1,    0,     0,   0   ],  # D6
  [0,    2,    2,    4,   0,     1,    5,    2,     0,   0   ],  # D7
  [6,    0,    1,    0,   4,     0,    0,    1,     2,   0   ],  # D8
  [2,    4,    3,    2,   0,     5,    3,    0,     0,   0   ],  # D9
  [1,    1,    4,    1,   1,     2,    4,    3,     0,   0   ],  # D10
  [8,    6,    2,    0,   5,     1,    0,    0,     3,   0   ],  # D11 — long doc
  [3,    2,    1,    3,   0,     0,    2,    5,     0,   0   ],  # D12
  [0,    3,    6,    2,   2,     4,    1,    0,     0,   1   ],  # D13
  [4,    4,    3,    1,   0,     3,    0,    0,     0,   2   ],  # D14
  [2,    0,    2,    5,   1,     2,    6,    1,     0,   0   ],  # D15
  [5,    3,    4,    0,   6,     0,    0,    0,     4,   0   ],  # D16 — neural-heavy
  [1,    5,    1,    2,   0,     6,    2,    0,     0,   0   ],  # D17
  [3,    1,    5,    3,   2,     1,    3,    2,     0,   1   ],  # D18
  [6,    4,    2,    0,   3,     2,    0,    0,     1,   0   ],  # D19
  [2,    3,    3,    4,   0,     3,    4,    0,     0,   3   ],  # D20 — quantum-heavy
]

doc_names = [f"D{i+1}" for i in range(20)]

# ── Query ─────────────────────────────────────────────────────────────────
query_terms = ["data", "neural", "search", "model"]

print(f"Corpus size : {len(tf_matrix)} documents")
print(f"Vocabulary  : {len(terms)} terms → {terms}")
print(f"Query       : {query_terms}")

Corpus size : 20 documents
Vocabulary  : 10 terms → ['data', 'model', 'search', 'rank', 'neural', 'query', 'index', 'sparse', 'bert', 'quantum']
Query       : ['data', 'neural', 'search', 'model']


---
## Step 3: Display the Input — Term-Frequency Table

Each cell shows how many times a term appears in that document.

In [3]:
# Build the TF DataFrame
df_tf = pd.DataFrame(tf_matrix, index=doc_names, columns=terms)

# Add a document-length column for clarity
df_tf.insert(0, "doc_length", df_tf.sum(axis=1))

print("=" * 70)
print("INPUT — Term-Frequency Matrix (rows=documents, columns=terms)")
print("=" * 70)
print(f"\nQuery: {query_terms}\n")
print(df_tf.to_string())
print(f"\nAverage document length: {df_tf['doc_length'].mean():.2f} terms")

INPUT — Term-Frequency Matrix (rows=documents, columns=terms)

Query: ['data', 'neural', 'search', 'model']

     doc_length  data  model  search  rank  neural  query  index  sparse  bert  quantum
D1           13     5      3       2     1       0      2      0       0     0        0
D2           13     3      2       4     0       1      1      2       0     0        0
D3           13     1      4       1     2       0      3      1       1     0        0
D4           14     7      1       3     0       2      0      0       0     1        0
D5           16     2      5       0     3       0      4      2       0     0        0
D6           19     4      3       5     1       3      2      1       0     0        0
D7           16     0      2       2     4       0      1      5       2     0        0
D8           14     6      0       1     0       4      0      0       1     2        0
D9           19     2      4       3     2       0      5      3       0     0        0
D10        

---
## Step 4: BM25 Implementation

### Parameters
- **k1 = 1.6** — controls term frequency saturation. Higher values reduce the saturation effect (TF keeps growing more linearly). Lower values (toward 0) make it binary-like.
- **b = 0.75** — controls length normalization. `b=1` = full normalization, `b=0` = no normalization.

### How it works step-by-step:
1. Compute document lengths and average document length (`avgdl`)
2. For each query term, compute its **IDF** across the corpus
3. For each document and each query term, compute the **BM25 term score**
4. Sum term scores per document → **final BM25 score**

In [4]:
# ── BM25 Parameters ───────────────────────────────────────────────────────
k1 = 1.6
b  = 0.75

N      = len(tf_matrix)                          # total number of documents
tf_arr = np.array(tf_matrix)                     # shape: (20, 10)
doc_lengths = tf_arr.sum(axis=1)                 # shape: (20,)
avgdl  = doc_lengths.mean()                      # scalar

term_index = {t: i for i, t in enumerate(terms)}

print(f"BM25 Parameters: k1={k1}, b={b}")
print(f"N={N} documents | avgdl={avgdl:.2f}")


def idf(term):
    """
    Robertson-Sparck Jones IDF:
        IDF(t) = ln( (N - n_t + 0.5) / (n_t + 0.5) + 1 )
    Adding 1 inside the log ensures IDF is always non-negative.
    """
    col = term_index[term]
    n_t = np.count_nonzero(tf_arr[:, col])   # docs containing the term
    return math.log((N - n_t + 0.5) / (n_t + 0.5) + 1)


def bm25_term_score(tf_val, doc_len, idf_val):
    """
    BM25 score contribution of one term in one document:
        idf * (tf * (k1+1)) / (tf + k1 * (1 - b + b * doc_len/avgdl))
    """
    numerator   = tf_val * (k1 + 1)
    denominator = tf_val + k1 * (1 - b + b * (doc_len / avgdl))
    return idf_val * (numerator / denominator) if denominator != 0 else 0.0


print("\nIDF values for query terms:")
for qt in query_terms:
    print(f"  IDF('{qt}') = {idf(qt):.4f}")

BM25 Parameters: k1=1.6, b=0.75
N=20 documents | avgdl=17.50

IDF values for query terms:
  IDF('data') = 0.1268
  IDF('neural') = 0.6022
  IDF('search') = 0.0741
  IDF('model') = 0.1268


---
## Step 5: Compute BM25 Scores

We compute, for each document, the BM25 contribution of **each query term** separately, then sum them for the total score.

In [5]:
# ── Compute per-term BM25 scores for every document ───────────────────────
results = {}   # term → list of per-doc scores

for qt in query_terms:
    col     = term_index[qt]
    idf_val = idf(qt)
    scores  = []
    for doc_i in range(N):
        tf_val  = tf_arr[doc_i, col]
        dl      = doc_lengths[doc_i]
        scores.append(bm25_term_score(tf_val, dl, idf_val))
    results[qt] = scores

# ── Total BM25 score = sum of per-term scores ─────────────────────────────
total_scores = np.array([results[qt] for qt in query_terms]).sum(axis=0)

print("BM25 scores computed for all 20 documents.")
print(f"Score range: [{total_scores.min():.4f}, {total_scores.max():.4f}]")

BM25 scores computed for all 20 documents.
Score range: [0.2987, 1.7582]


---
## Step 6: Display the Output — BM25 Score Table

The output table mirrors the input structure:
- **Rows** = documents (same order as input)
- **First column** = total BM25 score
- **Remaining columns** = per-term BM25 contribution (only for query terms)

In [6]:
# ── Build output DataFrame ─────────────────────────────────────────────────
df_out = pd.DataFrame(results, index=doc_names)
df_out.columns = [f"BM25({t})" for t in query_terms]
df_out.insert(0, "BM25_Score", total_scores)

print("=" * 70)
print("OUTPUT — BM25 Score Table (same document order as input)")
print("=" * 70)
print(f"\nQuery terms: {query_terms}")
print(f"Parameters : k1={k1}, b={b}\n")
print(df_out.to_string())

OUTPUT — BM25 Score Table (same document order as input)

Query terms: ['data', 'neural', 'search', 'model']
Parameters : k1=1.6, b=0.75

     BM25_Score  BM25(data)  BM25(neural)  BM25(search)  BM25(model)
D1       0.6094      0.2619        0.0000        0.1171       0.2304
D2       1.2596      0.2304        0.6833        0.1457       0.2003
D3       0.4770      0.1438        0.0000        0.0841       0.2491
D4       1.4801      0.2759        0.9319        0.1326       0.1396
D5       0.4421      0.1885        0.0000        0.0000       0.2536
D6       1.5839      0.2312        0.9987        0.1437       0.2102
D7       0.2987      0.0000        0.0000        0.1102       0.1885
D8       1.5187      0.2687        1.1684        0.0816       0.0000
D9       0.5321      0.1780        0.0000        0.1229       0.2312
D10      1.0056      0.1284        0.6102        0.1385       0.1284
D11      1.6984      0.2607        1.1004        0.0937       0.2437
D12      0.4855      0.2198       

---
## Step 7: Ranked Results

Documents are ranked from **highest to lowest** BM25 score.

In [7]:
df_ranked = df_out.sort_values("BM25_Score", ascending=False).copy()
df_ranked.insert(0, "Rank", range(1, N + 1))

print("=" * 70)
print("RANKED RESULTS — Documents sorted by BM25 score (high → low)")
print("=" * 70)
print(f"\nQuery: {query_terms}\n")
print(df_ranked.to_string(index=True))

RANKED RESULTS — Documents sorted by BM25 score (high → low)

Query: ['data', 'neural', 'search', 'model']

     Rank  BM25_Score  BM25(data)  BM25(neural)  BM25(search)  BM25(model)
D16     1      1.7582      0.2385        1.1878        0.1304       0.2014
D11     2      1.6984      0.2607        1.1004        0.0937       0.2437
D19     3      1.6125      0.2590        1.0135        0.1060       0.2340
D6      4      1.5839      0.2312        0.9987        0.1437       0.2102
D8      5      1.5187      0.2687        1.1684        0.0816       0.0000
D4      6      1.4801      0.2759        0.9319        0.1326       0.1396
D18     7      1.2766      0.2043        0.8154        0.1408       0.1160
D2      8      1.2596      0.2304        0.6833        0.1457       0.2003
D13     9      1.2060      0.0000        0.8456        0.1501       0.2102
D10    10      1.0056      0.1284        0.6102        0.1385       0.1284
D15    11      0.8613      0.1780        0.5793        0.1041      

---
## Step 8: Analysis & Interpretation

Let's inspect why BM25 ranked documents the way it did.

In [8]:
print("── IDF Analysis ──────────────────────────────────────────────────────")
print("Higher IDF = rarer term = greater discriminating power\n")

for qt in query_terms:
    col = term_index[qt]
    n_t = np.count_nonzero(tf_arr[:, col])
    print(f"  '{qt:8s}': appears in {n_t:2d}/{N} docs | IDF = {idf(qt):.4f}")

print("\n── Top 5 Documents ───────────────────────────────────────────────────")
top5 = df_ranked.head(5)
for doc, row in top5.iterrows():
    dl = int(doc_lengths[doc_names.index(doc)])
    print(f"  Rank {int(row['Rank'])}: {doc} | Score={row['BM25_Score']:.4f} | doc_length={dl}")

print("\n── Key BM25 Behaviors Illustrated ────────────────────────────────────")
print("  1. 'neural' has high IDF (rare term) → big score boost when present")
print("  2. 'data'/'model'/'search' are common → lower IDF, less discriminating")
print("  3. Long docs are penalized relative to the avgdl via the b=0.75 factor")
print("  4. Term saturation: tf=6 doesn't score 2× as much as tf=3 (unlike raw TF)")

── IDF Analysis ──────────────────────────────────────────────────────
Higher IDF = rarer term = greater discriminating power

  'data    ': appears in 18/20 docs | IDF = 0.1268
  'neural  ': appears in 11/20 docs | IDF = 0.6022
  'search  ': appears in 19/20 docs | IDF = 0.0741
  'model   ': appears in 18/20 docs | IDF = 0.1268

── Top 5 Documents ───────────────────────────────────────────────────
  Rank 1: D16 | Score=1.7582 | doc_length=22
  Rank 2: D11 | Score=1.6984 | doc_length=25
  Rank 3: D19 | Score=1.6125 | doc_length=18
  Rank 4: D6 | Score=1.5839 | doc_length=19
  Rank 5: D8 | Score=1.5187 | doc_length=14

── Key BM25 Behaviors Illustrated ────────────────────────────────────
  1. 'neural' has high IDF (rare term) → big score boost when present
  2. 'data'/'model'/'search' are common → lower IDF, less discriminating
  3. Long docs are penalized relative to the avgdl via the b=0.75 factor
  4. Term saturation: tf=6 doesn't score 2× as much as tf=3 (unlike raw TF)


---
## Summary

| Aspect | Value |
|---|---|
| Model | Okapi BM25 |
| Documents | 20 |
| Vocabulary size | 10 terms |
| Query | `data neural search model` |
| k1 | 1.6 |
| b | 0.75 |
| IDF formula | Robertson-Sparck Jones |

**BM25 outperforms TF-IDF** by:
1. **Saturating term frequency** — avoids over-rewarding repeated terms
2. **Normalizing for document length** — long documents don't unfairly dominate
3. **Using a robust IDF** — smoothed to handle edge cases (0 or N documents containing term)